In [23]:
import pandas as pd
import torch
import numpy as np
from models import Autoencoder
from utils import get_device, crear_datasets_proporcionales, estandarizar_columnas_no_binarias, separar_columnas_binarias

df = pd.read_csv("data/diabetes_012_health_indicators_BRFSS2015.csv")

# "Binarizamos" los datos, eliminando registros de pacientes con prediabetes
df = df[df["Diabetes_012"] != 1]
df["Diabetes_012"] = df["Diabetes_012"].replace(2, 1)

# Estandarizamos las columnas no binarias
df = estandarizar_columnas_no_binarias(df)
list_x_train, list_y_train, list_x_test, list_y_test, resumen_df = crear_datasets_proporcionales(df, "Diabetes_012")

device = get_device()
x0, x1, x2, x3 = list_x_test
print(resumen_df)


Dispositivo usado: cuda (NVIDIA GeForce GTX 970)

   Proporción  Positivos  Negativos  Total  % Positivos  % Negativos
0        0.00          0      70692  70692          0.0        100.0
1        0.10       7069      63623  70692         10.0         90.0
2        0.25      17673      53019  70692         25.0         75.0
3        0.50      35346      35346  70692         50.0         50.0


In [24]:
idx_cols_binarias, idx_cols_no_binarias = separar_columnas_binarias(x0)
print(idx_cols_no_binarias)

[ 3 13 14 15 18 19 20]


In [25]:
modelo = Autoencoder.load(path="models/autoencoder_fecha_00-11_25-11-25_lr_0.0001_conjunto_0.pth", device=device)
original = x0

reconstruido = modelo.predict(x=x0, device=device)

reconstruido[:, idx_cols_binarias] = np.round(np.abs(reconstruido[:, idx_cols_binarias]))

aciertos = np.sum(original == reconstruido)
total = original.size

print(f"{aciertos}/{total} valores reconstruidos correctamente. Esto es un {100 * aciertos / total:.2f}%")
print("\nArray original:")
print(original[0])
print("\nArray reconstruido:")
print(reconstruido[0])


Modelo cargado correctamente de 'models/autoencoder_fecha_00-11_25-11-25_lr_0.0001_conjunto_0.pth'
2217660/3745497 valores reconstruidos correctamente. Esto es un 59.21%

Array original:
[ 0.          0.          0.         -0.50633874  1.          0.
  0.          1.          0.          0.          0.          0.
  1.          0.46588173 -0.42814476 -0.48414917  0.          0.
 -0.3311125   0.96059071 -2.45224647]

Array reconstruido:
[ 0.          0.          1.         -0.72090966  0.          0.
  0.          1.          0.          0.          0.          1.
  0.         -0.10527401 -0.27918443 -0.361412    0.          0.
 -0.3131324   0.79450625 -2.18057   ]


c:\entornos-gpu\anomaly-detection-with-autoencoder\models.py:95: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  modelo = torch.load(path, map_location=device)


In [7]:
from utils import evaluar_anomalias, obtener_metricas

x0, x1, x2, x3 = list_x_test
y0, y1, y2, y3 = list_y_test

resultados = evaluar_anomalias(modelo, x0, y0, device, 1)
print(resultados)

metricas = obtener_metricas(**resultados)

print("Matriz de confusión (valores):")
print(metricas["matriz_confusion"])
print("\nMatriz de confusión (%):")
print(metricas["matriz_confusion_pct"])

print(f"\nAccuracy : {metricas['accuracy']}")
print(f"Precisión: {metricas['precision']}")
print(f"Recall   : {metricas['recall']}")
print(f"F1-Score : {metricas['f1_score']}")

{'TP': 19151, 'FN': 16195, 'TN': 82140, 'FP': 60871}
Matriz de confusión (valores):
[[82140 60871]
 [16195 19151]]

Matriz de confusión (%):
[[46.05 34.13]
 [ 9.08 10.74]]

Accuracy : 0.5679
Precisión: 0.2393
Recall   : 0.5418
F1-Score : 0.332
